# AiZynthFinder

Click the ▶ play button at the left of the **Installation** text below to install the application. The initial installation process may take a few minutes. Then run **Start application** cell.

1. Enter the target compound [SMILES][1] code.
3. Click the **Run Search** button to start the algorithm.
4. Once it stops serching, click the **Show Reactions** button.

[1]: https://en.wikipedia.org/wiki/Simplified_molecular-input_line-entry_system

In [ ]:
# 1. Install modern Python 3.13 wheels for all required dependencies
!pip install -q rdkit onnxruntime tables scipy pydantic deprecated jinja2 networkx pillow requests tqdm pyyaml

In [ ]:
# 2. Install AiZynthFinder itself without triggering old dependency builds
!pip install -q --ignore-requires-python --no-deps aizynthfinder

In [ ]:
# 3. Download the USPTO models and Zinc stock (takes ~30 seconds)
!mkdir -p data && download_public_data data

In [ ]:
!pip install -q rdchiral paretoset marshmallow jedi Reaction-Utils

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
import IPython
from google.colab import output
from rdkit import Chem
from rdkit.Chem import Draw
from aizynthfinder.aizynthfinder import AiZynthFinder

# 1. Initialize AiZynthFinder once
if "finder" not in globals():
    finder = AiZynthFinder(configfile="data/config.yml")
    finder.stock.select("zinc")
    finder.expansion_policy.select("uspto")
    finder.filter_policy.select("uspto")

# 2. Register Python callback triggered by the Save button
def _set_target_smiles(smiles_str):
    smiles_str = smiles_str.strip()
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None or not smiles_str:
        print(f"⚠️ Invalid or empty structure: '{smiles_str}'")
        return
    canon_smiles = Chem.MolToSmiles(mol)
    finder.target_smiles = canon_smiles
    print(f"✅ Successfully set finder.target_smiles = '{finder.target_smiles}'")
    display(Draw.MolToImage(mol, size=(320, 200)))

output.register_callback("notebook.set_target_smiles", _set_target_smiles)

# 3. Render interactive JSME Molecule Sketcher (No iframe / No CORS errors)
html_code = """
<div style="font-family: sans-serif; margin-bottom: 8px;">
  <b>Draw your target molecule below (or paste SMILES), then click the green button:</b>
</div>

<div id="jsme_container" style="width: 650px; height: 400px; border: 1px solid #ccc; border-radius: 6px;"></div>

<div style="margin-top: 10px; display: flex; align-items: center; gap: 10px; width: 650px;">
  <input id="manual-smiles" type="text" placeholder="Drawn SMILES will appear here (or paste SMILES)..."
         style="flex: 1; padding: 8px; font-family: monospace; font-size: 13px; border: 1px solid #ccc; border-radius: 4px;"
         value="Cc1nc(C2=CC=C(OCC(C)C)C(C#N)=C2)sc1C(=O)O" />
  <button id="save-smiles-btn" style="background-color: #28a745; color: white; border: none;
          padding: 9px 16px; font-size: 14px; font-weight: bold; border-radius: 5px; cursor: pointer; white-space: nowrap;">
    ✅ Set as finder.target_smiles
  </button>
</div>
<div id="smiles-status" style="margin-top: 6px; font-family: monospace; font-size: 13px; color: #155724;"></div>

<script>
// Dynamically load JSME inside the cell DOM
function initJSME() {
  if (typeof JSApplet !== "undefined") {
    window.jsmeApplet = new JSApplet.JSME("jsme_container", "650px", "400px");
    // Load default molecule into canvas
    window.jsmeApplet.readGenericMolecularInput("Cc1nc(C2=CC=C(OCC(C)C)C(C#N)=C2)sc1C(=O)O");
    // Sync canvas edits to the text box automatically
    window.jsmeApplet.setCallBack("AfterStructureModified", function(event) {
      const s = window.jsmeApplet.smiles();
      if (s) document.getElementById("manual-smiles").value = s;
    });
  }
}

window.jsmeOnLoad = function() { initJSME(); };

if (!document.getElementById("jsme-script")) {
  const script = document.createElement("script");
  script.id = "jsme-script";
  script.src = "https://jsme-editor.github.io/dist/jsme/jsme.nocache.js";
  document.head.appendChild(script);
} else {
  setTimeout(initJSME, 300);
}

document.getElementById('save-smiles-btn').onclick = function() {
  const statusEl = document.getElementById('smiles-status');
  let smiles = "";
  if (window.jsmeApplet && window.jsmeApplet.smiles()) {
    smiles = window.jsmeApplet.smiles();
    document.getElementById('manual-smiles').value = smiles;
  } else {
    smiles = document.getElementById('manual-smiles').value;
  }
  statusEl.innerText = "Setting finder.target_smiles = " + smiles + " ...";
  google.colab.kernel.invokeFunction('notebook.set_target_smiles', [smiles], {});
};
</script>
"""
IPython.display.display(IPython.display.HTML(html_code))

In [ ]:
finder.tree_search()
finder.build_routes()
stats = finder.extract_statistics()

print(f"Target: {finder.target_smiles}")
print(f"Solved: {stats['is_solved']} | Top Route Score: {finder.routes[0]['score']:.3f}")
finder.routes[0]["reaction_tree"].to_image()

In [ ]:
import logging
logging.getLogger("aizynthfinder").setLevel(logging.WARNING)

# Since finder.tree_search() and finder.build_routes() already completed,
# you can immediately inspect and plot the results (or uncomment to re-run):
finder.tree_search()
finder.build_routes()

stats = finder.extract_statistics()
top_score = finder.routes[0]["score"]["state score"]

print(f"Target: {finder.target_smiles}")
print(f"Solved: {stats['is_solved']} | Top Route Score: {top_score:.3f} | Total Routes: {len(finder.routes)}")

# Render the reaction tree image
finder.routes[0]["reaction_tree"].to_image()

In [ ]:
import logging
from IPython.display import display, HTML
from rdkit import Chem

# 1. Silence DEBUG logs
logging.getLogger("aizynthfinder").setLevel(logging.WARNING)
for handler in logging.getLogger("aizynthfinder").handlers:
    handler.setLevel(logging.WARNING)

# 2. Ensure up to 10 routes are extracted
finder.config.post_processing.min_routes = 10
finder.config.post_processing.max_routes = 10
finder.build_routes()

# Comprehensive RDKit SMARTS library for medicinal chemistry reactions
# We separate '|' into multiple patterns since RDKit doesn't parse it as OR globally
RAW_SMARTS = {
    "boronic": "[#5](O)O|[#5]1O[#6][#6]O1",
    "stannane": "[#50]",
    "zinc_magnesium": "[#12,#30]",
    "silicon_coupling": "[#14](F)|[#14](O)",
    "terminal_alkyne": "[CX2]#[CX2H1]",
    "internal_alkyne": "[CX2]#[CX2;!H1]",
    "alkene": "[CX3]=[CX3]",
    "aryl_halide_pseudohalide": "c[Cl,Br,I]|cOS(=O)(=O)C(F)(F)F",
    "alkyl_lg": "[CX4][Cl,Br,I]|[#6]OS(=O)(=O)[#6]",
    "carboxylic_acid": "[CX3](=O)[OX2H1]",
    "ester": "[CX3](=O)[OX2][#6]",
    "acyl_halide": "[CX3](=O)[Cl,Br,F]",
    "anhydride": "[CX3](=O)O[CX3](=O)",
    "amide": "[CX3](=O)[NX3]",
    "primary_amide": "[CX3](=O)[NX3H2]",
    "thioamide": "[#6](=[#16])[#7]",
    "amine_1_2": "[NX3;H2,H1;!$(NC=O);!$(NS=O)]",
    "tert_amine": "[NX3;H0;!$(NC=O);!$(NS=O)]",
    "aryl_amine": "c[NX3]",
    "nitro": "[$([NX3](=O)=O),$([NX3+](=O)[O-])]",
    "nitrile": "[CX2]#[NX1]",
    "azide": "[NX2]=[NX2+]=[NX1-]|[NX1-]=[NX2+]=[NX2]",
    "triazole": "c1ncnn1|c1nnn[nH]1",
    "tetrazole": "c1nnnn1",
    "thiazole": "c1ncsc1",
    "imidazole_oxazole": "c1nc[nH,o]c1",
    "indole": "c1ccc2[nH]ccc2c1",
    "aldehyde_ketone": "[CX3H1](=O)[#6]|[CX3](=O)([#6])[#6]",
    "alcohol": "[CX4][OX2H1]",
    "phenol": "c[OX2H1]",
    "ether": "[#6][OX2][#6]",
    "aryl_ether": "c[OX2][#6]",
    "epoxide": "C1OC1",
    "sulfonamide": "S(=O)(=O)[NX3]",
    "sulfonyl_chloride": "S(=O)(=O)[Cl]",
    "thiol": "[#6][SX2H1]",
    "thioether": "[#6][SX2][#6]",
    "sulfone_sulfoxide": "[#16X4](=O)(=O)|[#16X3](=O)",
    "boc": "C(=O)OC(C)(C)C",
    "cbz": "C(=O)OCc1ccccc1",
    "benzyl_ether_ester": "OCc1ccccc1",
    "silyl_ether": "[#8][#14]([#6])([#6])[#6]",
    "phosphorus_ylide_phosphonate": "[#15]",
}

MEDCHEM_SMARTS = {k: [Chem.MolFromSmarts(p) for p in v.split("|")] for k, v in RAW_SMARTS.items()}

def _has(mols, key):
    pats = MEDCHEM_SMARTS[key]
    return any(pat is not None and m is not None and m.HasSubstructMatch(pat) for m in mols for pat in pats)

def classify_reaction_step(rxn_node):
    metadata = getattr(rxn_node, "metadata", {}) or {}
    for key in ("classification", "reaction_name", "name", "type"):
        val = str(metadata.get(key, "")).strip()
        if val and "unrecognized" not in val.lower() and val != "0.0":
            return val

    rxn_smiles = getattr(rxn_node, "reaction_smiles", lambda: "")()
    if ">>" not in rxn_smiles:
        return "Functional Group Transformation"

    prod_str, react_str = rxn_smiles.split(">>")
    P = [Chem.MolFromSmiles(s) for s in prod_str.split(".") if s]
    R = [Chem.MolFromSmiles(s) for s in react_str.split(".") if s]

    # 1. Pd / Transition-Metal Cross-Couplings
    if _has(R, "boronic"):
        if _has(R, "aryl_halide_pseudohalide") or _has(R, "alkyl_lg"):
            return "Suzuki-Miyaura Cross-Coupling"
        if _has(R, "amine_1_2") or _has(R, "phenol"):
            return "Chan-Lam Coupling"
    if _has(R, "stannane") and _has(R, "aryl_halide_pseudohalide"):
        return "Stille Cross-Coupling"
    if _has(R, "terminal_alkyne") and _has(R, "aryl_halide_pseudohalide") and _has(P, "internal_alkyne"):
        return "Sonogashira Cross-Coupling"
    if _has(R, "aryl_halide_pseudohalide") and _has(R, "amine_1_2") and _has(P, "aryl_amine"):
        return "Buchwald-Hartwig Amination / Ullmann / SNAr Amination"
    if _has(R, "aryl_halide_pseudohalide") and _has(R, "alkene") and _has(P, "alkene"):
        return "Mizoroki-Heck Reaction"
    if _has(P, "boronic") and _has(R, "aryl_halide_pseudohalide"):
        return "Miyaura Borylation"
    if _has(R, "zinc_magnesium"):
        if _has(R, "aryl_halide_pseudohalide"):
            return "Kumada / Negishi Cross-Coupling"
        if _has(R, "aldehyde_ketone") or _has(R, "ester"):
            return "Grignard / Organometallic Nucleophilic Addition"

    # 2. Deprotections
    if _has(R, "boc") and not _has(P, "boc"):
        return "Boc Deprotection (Acidolysis)"
    if _has(R, "cbz") and not _has(P, "cbz"):
        return "Cbz Deprotection (Hydrogenolysis)"
    if _has(R, "silyl_ether") and not _has(P, "silyl_ether"):
        return "Silyl Ether Deprotection (Desilylation)"
    if _has(P, "carboxylic_acid") and _has(R, "ester") and not _has(R, "carboxylic_acid"):
        if _has(R, "benzyl_ether_ester"):
            return "Benzyl Ester Cleavage / Deprotection (Hydrolysis or Hydrogenolysis)"
        return "Ester Hydrolysis / Saponification (Deprotection)"
    if _has(P, "phenol") and _has(R, "aryl_ether") and not _has(R, "phenol"):
        return "Aryl Ether Cleavage / O-Dealkylation"

    # 3. Acylations, Amide & Sulfonamide Couplings, Esterifications
    if _has(P, "amide") and not _has(R, "amide") and _has(R, "amine_1_2"):
        if _has(R, "acyl_halide") or _has(R, "anhydride"):
            return "Schotten-Baumann Acylation / Amide Coupling"
        if _has(R, "carboxylic_acid"):
            return "Amide Bond Coupling (Carboxylic Acid + Amine)"
        if _has(R, "ester"):
            return "Ester Aminolysis"
    if _has(P, "sulfonamide") and _has(R, "sulfonyl_chloride") and _has(R, "amine_1_2"):
        return "Sulfonamide Formation (Sulfonylation of Amine)"
    if _has(P, "ester") and _has(R, "carboxylic_acid") and (_has(R, "alcohol") or _has(R, "phenol") or _has(R, "alkyl_lg")):
        return "Esterification (Fischer / Steglich / O-Alkylation)"

    # 4. Aminations & Reductions / Oxidations
    if (_has(P, "amine_1_2") or _has(P, "tert_amine")) and _has(R, "aldehyde_ketone") and _has(R, "amine_1_2"):
        return "Reductive Amination"
    if _has(P, "amine_1_2") and _has(R, "nitro") and not _has(P, "nitro"):
        return "Nitro Reduction to Amine"
    if _has(P, "alcohol") and _has(R, "aldehyde_ketone") and not _has(P, "aldehyde_ketone"):
        return "Carbonyl Reduction to Alcohol"
    if _has(P, "aldehyde_ketone") and _has(R, "alcohol") and not _has(R, "aldehyde_ketone"):
        return "Alcohol Oxidation to Carbonyl"
    if _has(P, "sulfone_sulfoxide") and _has(R, "thioether"):
        return "Sulfide Oxidation to Sulfoxide / Sulfone"

    # 5. Heterocycle Syntheses & Cycloadditions
    if _has(P, "thiazole") and not _has(R, "thiazole") and _has(R, "thioamide"):
        return "Hantzsch Thiazole Synthesis (Cyclization)"
    if _has(P, "triazole") and not _has(R, "triazole") and _has(R, "azide"):
        return "CuAAC 'Click' Azide-Alkyne Cycloaddition"
    if _has(P, "tetrazole") and not _has(R, "tetrazole") and _has(R, "nitrile"):
        return "[3+2] Azide-Nitrile Cycloaddition (Tetrazole Synthesis)"

    # 6. Nitriles, Ethers, Wittig & Substitutions
    if _has(P, "nitrile") and not _has(R, "nitrile"):
        if _has(R, "primary_amide"):
            return "Primary Amide Dehydration to Nitrile"
        if _has(R, "aryl_halide_pseudohalide"):
            return "Aryl Cyanation (Rosenmund-von Braun / Pd-Catalyzed)"
        if _has(R, "alkyl_lg"):
            return "Kolbe Nitrile Synthesis (SN2 Cyanation)"
    if _has(P, "alkene") and _has(R, "aldehyde_ketone") and _has(R, "phosphorus_ylide_phosphonate"):
        return "Wittig / Horner-Wadsworth-Emmons Olefination"
    if _has(P, "ether") and (_has(R, "alcohol") or _has(R, "phenol")):
        if _has(R, "alkyl_lg"):
            return "Williamson Ether Synthesis (O-Alkylation)"
        if _has(R, "alcohol"):
            return "Mitsunobu Reaction / Dehydrative Etherification"
    if (_has(P, "amine_1_2") or _has(P, "tert_amine")) and _has(R, "alkyl_lg") and _has(R, "amine_1_2"):
        return "N-Alkylation of Amine (SN2)"
    if _has(P, "aryl_halide_pseudohalide") and not _has(R, "aryl_halide_pseudohalide"):
        return "Electrophilic Aromatic Halogenation"

    return "Template-Guided Disconnection / Functional Group Interconversion"

# 4. Sort and display Top 10 routes
sorted_routes = sorted(
    finder.routes,
    key=lambda r: r["score"].get("state score", 0.0),
    reverse=True
)[:10]

stats = finder.extract_statistics()
print(f"Target SMILES: {finder.target_smiles}")
print(f"Search Solved: {stats['is_solved']} | Displaying Top {len(sorted_routes)} of {len(finder.routes)} Routes\n" + "="*85)

for idx, route in enumerate(sorted_routes, start=1):
    score_val = route["score"].get("state score", 0.0)
    rxn_tree = route["reaction_tree"]

    display(HTML(f"<h3 style='margin-bottom:4px; color:#1a73e8;'>Route #{idx} &mdash; State Score: {score_val:.4f} (Solved: {rxn_tree.is_solved})</h3>"))
    display(rxn_tree.to_image())

    # Reverse reactions list so Step 1..N matches Left-to-Right synthetic order in the diagram
    reactions_forward = list(reversed(list(rxn_tree.reactions())))
    print(f"Chemical Reaction Steps ({len(reactions_forward)} step{'s' if len(reactions_forward) != 1 else ''}, Left-to-Right Forward Order):")

    for step_idx, rxn_node in enumerate(reactions_forward, start=1):
        rxn_class = classify_reaction_step(rxn_node)
        meta = getattr(rxn_node, "metadata", {}) or {}
        template_code = meta.get("template_code", meta.get("template_hash", "N/A"))
        policy_prob = meta.get("policy_probability", None)
        prob_str = f" | Policy Prob: {policy_prob:.3f}" if isinstance(policy_prob, (int, float)) else ""

        try:
            retro_smi = rxn_node.reaction_smiles()
            # Flip Product>>Reactants to Reactants >> Product for intuitive forward reading
            if ">>" in retro_smi:
                prod_s, react_s = retro_smi.split(">>")
                forward_smi = f"{react_s} >> {prod_s}"
            else:
                forward_smi = retro_smi
        except Exception:
            forward_smi = "N/A"

        print(f"  Step {step_idx}: {rxn_class}  [Template ID: {template_code}{prob_str}]")
        print(f"          Forward Reaction: {forward_smi}")
    print("-" * 85)


In [ ]:
from rdkit import Chem

# Comprehensive RDKit SMARTS library for medicinal chemistry reactions
MEDCHEM_SMARTS = {
    "boronic": Chem.MolFromSmarts("[#5](O)O|[#5]1O[#6][#6]O1"),
    "stannane": Chem.MolFromSmarts("[#50]"),
    "zinc_magnesium": Chem.MolFromSmarts("[#12,#30]"),
    "silicon_coupling": Chem.MolFromSmarts("[#14](F)|[#14](O)"),
    "terminal_alkyne": Chem.MolFromSmarts("[CX2]#[CX2H1]"),
    "internal_alkyne": Chem.MolFromSmarts("[CX2]#[CX2;!H1]"),
    "alkene": Chem.MolFromSmarts("[CX3]=[CX3]"),
    "aryl_halide_pseudohalide": Chem.MolFromSmarts("c[Cl,Br,I]|cOS(=O)(=O)C(F)(F)F"),
    "alkyl_lg": Chem.MolFromSmarts("[CX4][Cl,Br,I]|[#6]OS(=O)(=O)[#6]"),
    "carboxylic_acid": Chem.MolFromSmarts("[CX3](=O)[OX2H1]"),
    "ester": Chem.MolFromSmarts("[CX3](=O)[OX2][#6]"),
    "acyl_halide": Chem.MolFromSmarts("[CX3](=O)[Cl,Br,F]"),
    "anhydride": Chem.MolFromSmarts("[CX3](=O)O[CX3](=O)"),
    "amide": Chem.MolFromSmarts("[CX3](=O)[NX3]"),
    "primary_amide": Chem.MolFromSmarts("[CX3](=O)[NX3H2]"),
    "thioamide": Chem.MolFromSmarts("[#6](=[#16])[#7]"),
    "amine_1_2": Chem.MolFromSmarts("[NX3;H2,H1;!$(NC=O);!$(NS=O)]"),
    "tert_amine": Chem.MolFromSmarts("[NX3;H0;!$(NC=O);!$(NS=O)]"),
    "aryl_amine": Chem.MolFromSmarts("c[NX3]"),
    "nitro": Chem.MolFromSmarts("[$([NX3](=O)=O),$([NX3+](=O)[O-])]"),
    "nitrile": Chem.MolFromSmarts("[CX2]#[NX1]"),
    "azide": Chem.MolFromSmarts("[NX2]=[NX2+]=[NX1-]|[NX1-]=[NX2+]=[NX2]"),
    "triazole": Chem.MolFromSmarts("c1ncnn1|c1nnn[nH]1"),
    "tetrazole": Chem.MolFromSmarts("c1nnnn1"),
    "thiazole": Chem.MolFromSmarts("c1ncsc1"),
    "imidazole_oxazole": Chem.MolFromSmarts("c1nc[nH,o]c1"),
    "indole": Chem.MolFromSmarts("c1ccc2[nH]ccc2c1"),
    "aldehyde_ketone": Chem.MolFromSmarts("[CX3H1](=O)[#6]|[CX3](=O)([#6])[#6]"),
    "alcohol": Chem.MolFromSmarts("[CX4][OX2H1]"),
    "phenol": Chem.MolFromSmarts("c[OX2H1]"),
    "ether": Chem.MolFromSmarts("[#6][OX2][#6]"),
    "aryl_ether": Chem.MolFromSmarts("c[OX2][#6]"),
    "epoxide": Chem.MolFromSmarts("C1OC1"),
    "sulfonamide": Chem.MolFromSmarts("S(=O)(=O)[NX3]"),
    "sulfonyl_chloride": Chem.MolFromSmarts("S(=O)(=O)[Cl]"),
    "thiol": Chem.MolFromSmarts("[#6][SX2H1]"),
    "thioether": Chem.MolFromSmarts("[#6][SX2][#6]"),
    "sulfone_sulfoxide": Chem.MolFromSmarts("[#16X4](=O)(=O)|[#16X3](=O)"),
    "boc": Chem.MolFromSmarts("C(=O)OC(C)(C)C"),
    "cbz": Chem.MolFromSmarts("C(=O)OCc1ccccc1"),
    "benzyl_ether_ester": Chem.MolFromSmarts("OCc1ccccc1"),
    "silyl_ether": Chem.MolFromSmarts("[#8][#14]([#6])([#6])[#6]"),
    "phosphorus_ylide_phosphonate": Chem.MolFromSmarts("[#15]"),
}

def _has(mols, key):
    pat = MEDCHEM_SMARTS[key]
    return any(m is not None and m.HasSubstructMatch(pat) for m in mols)

def classify_reaction_step(rxn_node):
    metadata = getattr(rxn_node, "metadata", {}) or {}
    for key in ("classification", "reaction_name", "name", "type"):
        val = str(metadata.get(key, "")).strip()
        if val and "unrecognized" not in val.lower() and val != "0.0":
            return val

    rxn_smiles = getattr(rxn_node, "reaction_smiles", lambda: "")()
    if ">>" not in rxn_smiles:
        return "Functional Group Transformation"

    prod_str, react_str = rxn_smiles.split(">>")
    P = [Chem.MolFromSmiles(s) for s in prod_str.split(".") if s]
    R = [Chem.MolFromSmiles(s) for s in react_str.split(".") if s]

    # 1. Pd / Transition-Metal Cross-Couplings
    if _has(R, "boronic"):
        if _has(R, "aryl_halide_pseudohalide") or _has(R, "alkyl_lg"):
            return "Suzuki-Miyaura Cross-Coupling"
        if _has(R, "amine_1_2") or _has(R, "phenol"):
            return "Chan-Lam Coupling"
    if _has(R, "stannane") and _has(R, "aryl_halide_pseudohalide"):
        return "Stille Cross-Coupling"
    if _has(R, "terminal_alkyne") and _has(R, "aryl_halide_pseudohalide") and _has(P, "internal_alkyne"):
        return "Sonogashira Cross-Coupling"
    if _has(R, "aryl_halide_pseudohalide") and _has(R, "amine_1_2") and _has(P, "aryl_amine"):
        return "Buchwald-Hartwig Amination / Ullmann / SNAr Amination"
    if _has(R, "aryl_halide_pseudohalide") and _has(R, "alkene") and _has(P, "alkene"):
        return "Mizoroki-Heck Reaction"
    if _has(P, "boronic") and _has(R, "aryl_halide_pseudohalide"):
        return "Miyaura Borylation"
    if _has(R, "zinc_magnesium"):
        if _has(R, "aryl_halide_pseudohalide"):
            return "Kumada / Negishi Cross-Coupling"
        if _has(R, "aldehyde_ketone") or _has(R, "ester"):
            return "Grignard / Organometallic Nucleophilic Addition"

    # 2. Deprotections
    if _has(R, "boc") and not _has(P, "boc"):
        return "Boc Deprotection (Acidolysis)"
    if _has(R, "cbz") and not _has(P, "cbz"):
        return "Cbz Deprotection (Hydrogenolysis)"
    if _has(R, "silyl_ether") and not _has(P, "silyl_ether"):
        return "Silyl Ether Deprotection (Desilylation)"
    if _has(P, "carboxylic_acid") and _has(R, "ester") and not _has(R, "carboxylic_acid"):
        if _has(R, "benzyl_ether_ester"):
            return "Benzyl Ester Cleavage / Deprotection (Hydrolysis or Hydrogenolysis)"
        return "Ester Hydrolysis / Saponification (Deprotection)"
    if _has(P, "phenol") and _has(R, "aryl_ether") and not _has(R, "phenol"):
        return "Aryl Ether Cleavage / O-Dealkylation"

    # 3. Acylations, Amide & Sulfonamide Couplings, Esterifications
    if _has(P, "amide") and not _has(R, "amide") and _has(R, "amine_1_2"):
        if _has(R, "acyl_halide") or _has(R, "anhydride"):
            return "Schotten-Baumann Acylation / Amide Coupling"
        if _has(R, "carboxylic_acid"):
            return "Amide Bond Coupling (Carboxylic Acid + Amine)"
        if _has(R, "ester"):
            return "Ester Aminolysis"
    if _has(P, "sulfonamide") and _has(R, "sulfonyl_chloride") and _has(R, "amine_1_2"):
        return "Sulfonamide Formation (Sulfonylation of Amine)"
    if _has(P, "ester") and _has(R, "carboxylic_acid") and (_has(R, "alcohol") or _has(R, "phenol") or _has(R, "alkyl_lg")):
        return "Esterification (Fischer / Steglich / O-Alkylation)"

    # 4. Aminations & Reductions / Oxidations
    if (_has(P, "amine_1_2") or _has(P, "tert_amine")) and _has(R, "aldehyde_ketone") and _has(R, "amine_1_2"):
        return "Reductive Amination"
    if _has(P, "amine_1_2") and _has(R, "nitro") and not _has(P, "nitro"):
        return "Nitro Reduction to Amine"
    if _has(P, "alcohol") and _has(R, "aldehyde_ketone") and not _has(P, "aldehyde_ketone"):
        return "Carbonyl Reduction to Alcohol"
    if _has(P, "aldehyde_ketone") and _has(R, "alcohol") and not _has(R, "aldehyde_ketone"):
        return "Alcohol Oxidation to Carbonyl"
    if _has(P, "sulfone_sulfoxide") and _has(R, "thioether"):
        return "Sulfide Oxidation to Sulfoxide / Sulfone"

    # 5. Heterocycle Syntheses & Cycloadditions
    if _has(P, "thiazole") and not _has(R, "thiazole") and _has(R, "thioamide"):
        return "Hantzsch Thiazole Synthesis (Cyclization)"
    if _has(P, "triazole") and not _has(R, "triazole") and _has(R, "azide"):
        return "CuAAC 'Click' Azide-Alkyne Cycloaddition"
    if _has(P, "tetrazole") and not _has(R, "tetrazole") and _has(R, "nitrile"):
        return "[3+2] Azide-Nitrile Cycloaddition (Tetrazole Synthesis)"

    # 6. Nitriles, Ethers, Wittig & Substitutions
    if _has(P, "nitrile") and not _has(R, "nitrile"):
        if _has(R, "primary_amide"):
            return "Primary Amide Dehydration to Nitrile"
        if _has(R, "aryl_halide_pseudohalide"):
            return "Aryl Cyanation (Rosenmund-von Braun / Pd-Catalyzed)"
        if _has(R, "alkyl_lg"):
            return "Kolbe Nitrile Synthesis (SN2 Cyanation)"
    if _has(P, "alkene") and _has(R, "aldehyde_ketone") and _has(R, "phosphorus_ylide_phosphonate"):
        return "Wittig / Horner-Wadsworth-Emmons Olefination"
    if _has(P, "ether") and (_has(R, "alcohol") or _has(R, "phenol")):
        if _has(R, "alkyl_lg"):
            return "Williamson Ether Synthesis (O-Alkylation)"
        if _has(R, "alcohol"):
            return "Mitsunobu Reaction / Dehydrative Etherification"
    if (_has(P, "amine_1_2") or _has(P, "tert_amine")) and _has(R, "alkyl_lg") and _has(R, "amine_1_2"):
        return "N-Alkylation of Amine (SN2)"
    if _has(P, "aryl_halide_pseudohalide") and not _has(R, "aryl_halide_pseudohalide"):
        return "Electrophilic Aromatic Halogenation"

    return "Template-Guided Disconnection / Functional Group Interconversion"

In [ ]:
import os
import json
import logging
from IPython.display import display, HTML
from pydantic import BaseModel
from rdkit import Chem

# 1. Silence DEBUG logs
logging.getLogger("aizynthfinder").setLevel(logging.WARNING)
for handler in logging.getLogger("aizynthfinder").handlers:
    handler.setLevel(logging.WARNING)

# 2. Ensure up to 10 routes are extracted
finder.config.post_processing.min_routes = 10
finder.config.post_processing.max_routes = 10
finder.build_routes()

# 3. Fixed RDKit SMARTS Library (Note: Cbz now strictly requires Nitrogen '[NX3]C(=O)OCc1ccccc1')
MEDCHEM_SMARTS = {
    "boronic": Chem.MolFromSmarts("[#5](O)O|[#5]1O[#6][#6]O1"),
    "stannane": Chem.MolFromSmarts("[#50]"),
    "zinc_magnesium": Chem.MolFromSmarts("[#12,#30]"),
    "terminal_alkyne": Chem.MolFromSmarts("[CX2]#[CX2H1]"),
    "internal_alkyne": Chem.MolFromSmarts("[CX2]#[CX2;!H1]"),
    "alkene": Chem.MolFromSmarts("[CX3]=[CX3]"),
    "aryl_halide_pseudohalide": Chem.MolFromSmarts("c[Cl,Br,I]|cOS(=O)(=O)C(F)(F)F"),
    "alkyl_lg": Chem.MolFromSmarts("[CX4][Cl,Br,I]|[#6]OS(=O)(=O)[#6]"),
    "carboxylic_acid": Chem.MolFromSmarts("[CX3](=O)[OX2H1]"),
    "ester": Chem.MolFromSmarts("[#6][CX3](=O)[OX2][#6]"),
    "benzyl_ester": Chem.MolFromSmarts("[#6][CX3](=O)[OX2]Cc1ccccc1"),
    "acyl_halide": Chem.MolFromSmarts("[CX3](=O)[Cl,Br,F]"),
    "anhydride": Chem.MolFromSmarts("[CX3](=O)O[CX3](=O)"),
    "amide": Chem.MolFromSmarts("[#6][CX3](=O)[NX3]"),
    "primary_amide": Chem.MolFromSmarts("[CX3](=O)[NX3H2]"),
    "thioamide": Chem.MolFromSmarts("[#6](=[#16])[#7]"),
    "amine_1_2": Chem.MolFromSmarts("[NX3;H2,H1;!$(NC=O);!$(NS=O)]"),
    "tert_amine": Chem.MolFromSmarts("[NX3;H0;!$(NC=O);!$(NS=O)]"),
    "aryl_amine": Chem.MolFromSmarts("c[NX3]"),
    "nitro": Chem.MolFromSmarts("[$([NX3](=O)=O),$([NX3+](=O)[O-])]"),
    "nitrile": Chem.MolFromSmarts("[CX2]#[NX1]"),
    "azide": Chem.MolFromSmarts("[NX2]=[NX2+]=[NX1-]|[NX1-]=[NX2+]=[NX2]"),
    "triazole": Chem.MolFromSmarts("c1ncnn1|c1nnn[nH]1"),
    "tetrazole": Chem.MolFromSmarts("c1nnnn1"),
    "thiazole": Chem.MolFromSmarts("c1ncsc1"),
    "aldehyde_ketone": Chem.MolFromSmarts("[CX3H1](=O)[#6]|[CX3](=O)([#6])[#6]"),
    "alcohol": Chem.MolFromSmarts("[CX4][OX2H1]"),
    "phenol": Chem.MolFromSmarts("c[OX2H1]"),
    "ether": Chem.MolFromSmarts("[#6][OX2][#6]"),
    "aryl_ether": Chem.MolFromSmarts("c[OX2][CX4]"),
    "sulfonamide": Chem.MolFromSmarts("S(=O)(=O)[NX3]"),
    "sulfonyl_chloride": Chem.MolFromSmarts("S(=O)(=O)[Cl]"),
    "thioether": Chem.MolFromSmarts("[#6][SX2][#6]"),
    "sulfone_sulfoxide": Chem.MolFromSmarts("[#16X4](=O)(=O)|[#16X3](=O)"),
    "boc": Chem.MolFromSmarts("[NX3]C(=O)OC(C)(C)C"),
    "cbz": Chem.MolFromSmarts("[NX3]C(=O)OCc1ccccc1"),
    "silyl_ether": Chem.MolFromSmarts("[#8][#14]([#6])([#6])[#6]"),
    "phosphorus_ylide_phosphonate": Chem.MolFromSmarts("[#15]"),
}

def _has(mols, key):
    pat = MEDCHEM_SMARTS[key]
    return any(m is not None and m.HasSubstructMatch(pat) for m in mols)

def rdkit_classify_step(forward_smi):
    """Fast local RDKit heuristic classifier for Reactants >> Product."""
    if ">>" not in forward_smi:
        return "Functional Group Transformation", "Standard literature conditions"
    react_str, prod_str = [s.strip() for s in forward_smi.split(">>")]
    R = [Chem.MolFromSmiles(s) for s in react_str.split(".") if s]
    P = [Chem.MolFromSmiles(s) for s in prod_str.split(".") if s]

    if _has(R, "boronic"):
        if _has(R, "aryl_halide_pseudohalide") or _has(R, "alkyl_lg"):
            return "Suzuki-Miyaura Cross-Coupling", "Pd(dppf)Cl2 or Pd(PPh3)4, K2CO3, Dioxane/H2O, 80-100 °C"
        if _has(R, "amine_1_2") or _has(R, "phenol"):
            return "Chan-Lam Coupling", "Cu(OAc)2, Pyridine or Et3N, DCM, O2 (air), RT"
    if _has(R, "stannane") and _has(R, "aryl_halide_pseudohalide"):
        return "Stille Cross-Coupling", "Pd(PPh3)4, LiCl, DMF or Dioxane, 90 °C"
    if _has(R, "terminal_alkyne") and _has(R, "aryl_halide_pseudohalide") and _has(P, "internal_alkyne"):
        return "Sonogashira Cross-Coupling", "Pd(PPh3)2Cl2, CuI, Et3N, THF/DMF, RT-60 °C"
    if _has(R, "aryl_halide_pseudohalide") and _has(R, "amine_1_2") and _has(P, "aryl_amine"):
        return "Buchwald-Hartwig Amination / SNAr", "Pd2(dba)3, XPhos/BINAP, NaOtBu, Toluene, 100 °C"
    if _has(P, "boronic") and _has(R, "aryl_halide_pseudohalide"):
        return "Miyaura Borylation", "B2pin2, Pd(dppf)Cl2, KOAc, Dioxane, 90 °C"
    if _has(R, "boc") and not _has(P, "boc"):
        return "N-Boc Deprotection (Acidolysis)", "TFA / DCM or 4M HCl in Dioxane, RT"
    if _has(R, "cbz") and not _has(P, "cbz"):
        return "N-Cbz Deprotection (Hydrogenolysis)", "H2, Pd/C, MeOH or EtOAc, RT"
    if _has(P, "carboxylic_acid") and _has(R, "ester") and not _has(R, "carboxylic_acid"):
        if _has(R, "benzyl_ester"):
            return "Benzyl Ester Deprotection (Hydrogenolysis / Hydrolysis)", "H2, Pd/C in MeOH/THF (or LiOH, THF/H2O)"
        return "Ester Hydrolysis / Saponification", "LiOH or NaOH, THF / MeOH / H2O, RT-50 °C"
    if _has(P, "thiazole") and not _has(R, "thiazole") and _has(R, "thioamide"):
        return "Hantzsch Thiazole Synthesis (Cyclization)", "EtOH or DMF, reflux (70-90 °C)"
    if _has(P, "amide") and not _has(R, "amide") and _has(R, "amine_1_2"):
        if _has(R, "acyl_halide"):
            return "Schotten-Baumann Amide Coupling", "Et3N or DIPEA, DCM, 0 °C to RT"
        if _has(R, "carboxylic_acid"):
            return "Amide Bond Coupling", "HATU or EDC/HOBt, DIPEA, DMF, RT"
    if _has(P, "aryl_ether") and _has(R, "phenol"):
        return "Williamson Ether Synthesis / O-Alkylation", "K2CO3 or Cs2CO3, DMF or MeCN, 60-80 °C"
    if _has(P, "nitrile") and not _has(R, "nitrile"):
        if _has(R, "primary_amide"):
            return "Primary Amide Dehydration to Nitrile", "TFAA, Et3N, DCM or POCl3"
        if _has(R, "aryl_halide_pseudohalide"):
            return "Pd-Catalyzed Aryl Cyanation", "Zn(CN)2, Pd(PPh3)4, DMF, 100 °C"
    if (_has(P, "amine_1_2") or _has(P, "tert_amine")) and _has(R, "aldehyde_ketone") and _has(R, "amine_1_2"):
        return "Reductive Amination", "NaBH(OAc)3 or NaBH3CN, AcOH, DCE/MeOH, RT"
    return "Template-Guided Disconnection", "Standard literature conditions"

# 4. Optional Batched Gemini API Classifier (100% Organic Chemistry Coverage + Conditions)
USE_GEMINI_CLASSIFIER = True  # Automatically falls back to RDKit if no API key is set

class StepClassification(BaseModel):
    forward_smiles: str
    reaction_name: str
    typical_conditions: str

class BatchReactionResponse(BaseModel):
    reactions: list[StepClassification]

def batch_classify_with_gemini(unique_forward_smiles):
    """Classifies all unique reaction steps in a single fast Gemini API call."""
    api_key = None
    try:
        from google.colab import userdata
        api_key = userdata.get("KDD_GEMINI_API_KEY")
    except Exception:
        api_key = os.environ.get("KDD_GEMINI_API_KEY")

    if not api_key or not USE_GEMINI_CLASSIFIER:
        return {}

    try:
        from google import genai
        client = genai.Client(api_key=api_key)
        prompt = (
            "You are an expert medicinal and process chemist. For each forward reaction SMILES "
            "(Reactants >> Product) below, identify the specific Named Reaction or precise chemical "
            "transformation class (e.g., 'Suzuki-Miyaura Cross-Coupling', 'Hantzsch Thiazole Synthesis', "
            "'Benzyl Ester Hydrogenolysis / Saponification', 'Williamson Ether Synthesis') and list "
            "concise, realistic lab reagents/catalysts/solvents/temperature.\n\n"
            + "\n".join(f"- {smi}" for smi in unique_forward_smiles)
        )
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
            config={
                "response_mime_type": "application/json",
                "response_schema": BatchReactionResponse,
                "temperature": 0.1,
            },
        )
        parsed = json.loads(response.text)
        return {
            item["forward_smiles"]: (item["reaction_name"], item["typical_conditions"])
            for item in parsed.get("reactions", [])
        }
    except Exception as e:
        print(f"ℹ️ Gemini API not active ({e}); using local RDKit SMARTS classifier.")
        return {}

# 5. Collect Top 10 Routes & Batch-Classify Unique Steps
sorted_routes = sorted(
    finder.routes,
    key=lambda r: r["score"].get("state score", 0.0),
    reverse=True
)[:10]

# Gather all unique forward reaction SMILES across the Top 10 routes
unique_steps = []
for route in sorted_routes:
    for rxn_node in route["reaction_tree"].reactions():
        retro_smi = rxn_node.reaction_smiles()
        if ">>" in retro_smi:
            prod_s, react_s = retro_smi.split(">>")
            fwd_smi = f"{react_s} >> {prod_s}"
            if fwd_smi not in unique_steps:
                unique_steps.append(fwd_smi)

gemini_lookup = batch_classify_with_gemini(unique_steps)

# 6. Render Top 10 Routes with Reaction Names & Conditions
stats = finder.extract_statistics()
classifier_mode = "Gemini 2.5 Flash + RDKit SMARTS" if gemini_lookup else "Local RDKit MedChem SMARTS"
print(f"Target SMILES: {finder.target_smiles} | Classifier: {classifier_mode}")
print(f"Search Solved: {stats['is_solved']} | Displaying Top {len(sorted_routes)} of {len(finder.routes)} Routes\n" + "="*90)

for idx, route in enumerate(sorted_routes, start=1):
    score_val = route["score"].get("state score", 0.0)
    rxn_tree = route["reaction_tree"]

    display(HTML(f"<h3 style='margin-bottom:4px; color:#1a73e8;'>Route #{idx} &mdash; State Score: {score_val:.4f} (Solved: {rxn_tree.is_solved})</h3>"))
    display(rxn_tree.to_image())

    reactions_forward = list(reversed(list(rxn_tree.reactions())))
    print(f"Chemical Reaction Steps ({len(reactions_forward)} step{'s' if len(reactions_forward) != 1 else ''}, Left-to-Right Forward Order):")

    for step_idx, rxn_node in enumerate(reactions_forward, start=1):
        meta = getattr(rxn_node, "metadata", {}) or {}
        template_code = meta.get("template_code", meta.get("template_hash", "N/A"))
        policy_prob = meta.get("policy_probability", None)
        prob_str = f" | Policy Prob: {policy_prob:.3f}" if isinstance(policy_prob, (int, float)) else ""

        retro_smi = rxn_node.reaction_smiles()
        prod_s, react_s = retro_smi.split(">>") if ">>" in retro_smi else ("", retro_smi)
        forward_smi = f"{react_s} >> {prod_s}"

        # Look up Gemini classification first; fall back to local RDKit SMARTS
        if forward_smi in gemini_lookup:
            rxn_name, conditions = gemini_lookup[forward_smi]
        else:
            rxn_name, conditions = rdkit_classify_step(forward_smi)

        print(f"  Step {step_idx}: {rxn_name}  [Template ID: {template_code}{prob_str}]")
        print(f"          Suggested Conditions: {conditions}")
        print(f"          Forward SMILES:       {forward_smi}")
    print("-" * 90)

In [ ]:
import os
import html
import markdown
import IPython
from google.colab import output, userdata
from google import genai
from google.genai import types

# 1. Build a context summary of the current Target & Top 10 AiZynthFinder Routes
def _build_retrosynthesis_context():
    if "finder" not in globals() or not getattr(finder, "target_smiles", None):
        return "No target molecule or routes have been generated in AiZynthFinder yet."

    lines = [f"Target Molecule SMILES: {finder.target_smiles}\n"]
    sorted_r = sorted(
        finder.routes,
        key=lambda r: r["score"].get("state score", 0.0),
        reverse=True
    )[:10]

    for idx, route in enumerate(sorted_r, start=1):
        score_val = route["score"].get("state score", 0.0)
        rxn_tree = route["reaction_tree"]
        lines.append(f"### Route #{idx} (State Score: {score_val:.4f}, Solved: {rxn_tree.is_solved})")
        reactions_forward = list(reversed(list(rxn_tree.reactions())))
        for s_idx, rxn_node in enumerate(reactions_forward, start=1):
            meta = getattr(rxn_node, "metadata", {}) or {}
            t_id = meta.get("template_code", meta.get("template_hash", "N/A"))
            retro_smi = rxn_node.reaction_smiles()
            prod_s, react_s = retro_smi.split(">>") if ">>" in retro_smi else ("", retro_smi)
            fwd_smi = f"{react_s} >> {prod_s}"
            name_hint = gemini_lookup.get(fwd_smi, ("Reaction Step", ""))[0] if "gemini_lookup" in globals() else "Step"
            lines.append(f"  - Step {s_idx} ({name_hint}, Template {t_id}): {fwd_smi}")
    return "\n".join(lines)

# 2. Persistent Multi-Turn Chat State
_CHAT_HISTORY = []  # Stores types.Content objects for multi-turn conversation

SYSTEM_INSTRUCTION = f"""You are an expert medicinal and process chemist assistant integrated into an AiZynthFinder retrosynthesis notebook.
Your job is to help the user find accurate, peer-reviewed and patent-backed reaction conditions (reagents, catalysts, ligands, solvents, temperatures, reaction times, and workup/selectivity notes) for the retrosynthetic routes generated below.

CRITICAL INSTRUCTIONS:
1. Always use Google Search grounding to verify literature precedents, journal references (e.g., J. Med. Chem., Org. Process Res. Dev., J. Org. Chem., Tetrahedron Lett.), and patent procedures (USPTO / WO patents) for the exact target molecule or closely related scaffolds.
2. Provide specific stoichiometric equivalents, catalysts, solvents, and temperatures where reported.
3. Explicitly cite your references inline (e.g., Author et al., Journal Year, Volume, Pages / DOI or Patent Number) and list full citations at the end of your response.
4. Flag any chemoselectivity, regioselectivity, or functional group compatibility risks (e.g., nitrile hydrolysis during ester saponification, SNAr ortho-nitrile activation).
5. Output chemical formulas as plain text (e.g., Pd(PPh3)4, Na2CO3). DO NOT use LaTeX, Markdown math formatting, or MathJax (no $ signs, no \text{{}} blocks, no subscript notation) anywhere in your response.

CURRENT AIZYNTHFINDER ROUTES IN NOTEBOOK:
{_build_retrosynthesis_context()}
"""

def _send_chat_message(user_text, selected_model, use_search):
    global _CHAT_HISTORY
    user_text = (user_text or "").strip()
    if not user_text:
        return IPython.display.JSON({"error": "Empty message"})

    api_key = None
    try:
        api_key = userdata.get("KDD_GEMINI_API_KEY")
    except Exception:
        api_key = os.environ.get("KDD_GEMINI_API_KEY")

    if not api_key:
        return IPython.display.JSON({
            "error": "Missing GEMINI_API_KEY. Please add GEMINI_API_KEY to Google Colab Secrets (🔑 icon on the left sidebar)."
        })

    client = genai.Client(api_key=api_key)
    _CHAT_HISTORY.append(types.Content(role="user", parts=[types.Part.from_text(text=user_text)]))

    # Configure Google Search Grounding for live literature & patent lookup
    tools = [types.Tool(google_search=types.GoogleSearch())] if use_search else []
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        tools=tools,
        temperature=0.2,
    )

    # Attempt selected model first (e.g. flash-3.8); gracefully fall back if model alias isn't live on endpoint
    candidate_models = [selected_model, "gemini-3-flash-preview", "gemini-2.5-flash"]
    used_model = selected_model
    response = None
    last_err = None

    for m in dict.fromkeys(candidate_models):
        try:
            response = client.models.generate_content(
                model=m,
                contents=_CHAT_HISTORY,
                config=config,
            )
            used_model = m
            break
        except Exception as e:
            last_err = str(e)

    if response is None:
        _CHAT_HISTORY.pop()  # Remove failed turn
        return IPython.display.JSON({"error": f"API Error: {last_err}"})

    reply_text = response.text or "(No text returned)"
    _CHAT_HISTORY.append(types.Content(role="model", parts=[types.Part.from_text(text=reply_text)]))

    # Extract grounded web/literature citations if returned by Google Search tool
    sources_html = ""
    try:
        cand = response.candidates[0]
        gm = getattr(cand, "grounding_metadata", None)
        chunks = getattr(gm, "grounding_chunks", None) if gm else None
        if chunks:
            links = []
            for c in chunks:
                web = getattr(c, "web", None)
                if web and getattr(web, "uri", None):
                    title = html.escape(getattr(web, "title", None) or web.uri)
                    links.append(f"<li><a href='{web.uri}' target='_blank' style='color:#1a73e8;'>{title}</a></li>")
            if links:
                sources_html = (
                    "<div style='margin-top:10px; padding-top:8px; border-top:1px solid #e0e0e0; font-size:12px;'>"
                    "<b>📚 Grounded Literature & Web Sources:</b><ul style='margin:4px 0 0 18px; padding:0;'>"
                    + "".join(dict.fromkeys(links))
                    + "</ul></div>"
                )
    except Exception:
        pass

    note = ""
    if used_model != selected_model:
        note = f"<div style='font-size:11px; color:#b06000; margin-bottom:4px;'>ℹ️ Note: '{selected_model}' routed to active endpoint '{used_model}'.</div>"

    rendered_md = markdown.markdown(reply_text, extensions=["tables", "fenced_code"])
    return IPython.display.JSON({
        "html": note + rendered_md + sources_html,
        "model": used_model
    })

def _reset_chat_history():
    global _CHAT_HISTORY
    _CHAT_HISTORY = []
    return IPython.display.JSON({"status": "cleared"})

output.register_callback("notebook.gemini_retro_chat", _send_chat_message)
output.register_callback("notebook.gemini_retro_reset", _reset_chat_history)

# 3. Render Interactive Chat UI inside the Colab Cell Output
chat_ui_html = """
<style>
  .retro-chat-box {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
    border: 1px solid #dadce0;
    border-radius: 10px;
    background: #ffffff;
    max-width: 960px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.06);
    overflow: hidden;
  }
  .retro-chat-header {
    background: #f8f9fa;
    padding: 12px 16px;
    border-bottom: 1px solid #dadce0;
    display: flex;
    flex-wrap: wrap;
    align-items: center;
    justify-content: space-between;
    gap: 10px;
  }
  .retro-chat-messages {
    height: 440px;
    overflow-y: auto;
    padding: 16px;
    background: #fcfcfc;
    display: flex;
    flex-direction: column;
    gap: 12px;
  }
  .msg-user {
    align-self: flex-end;
    background: #e8f0fe;
    color: #174ea6;
    padding: 10px 14px;
    border-radius: 14px 14px 2px 14px;
    max-width: 78%;
    font-size: 14px;
    line-height: 1.45;
  }
  .msg-model {
    align-self: flex-start;
    background: #ffffff;
    color: #202124;
    border: 1px solid #e0e0e0;
    padding: 12px 16px;
    border-radius: 14px 14px 14px 2px;
    max-width: 88%;
    font-size: 14px;
    line-height: 1.55;
  }
  .msg-model table {
    border-collapse: collapse;
    margin: 8px 0;
    font-size: 13px;
  }
  .msg-model th, .msg-model td {
    border: 1px solid #ccc;
    padding: 6px 10px;
  }
  .retro-chat-input-area {
    padding: 12px 16px;
    border-top: 1px solid #dadce0;
    background: #f8f9fa;
    display: flex;
    gap: 10px;
    align-items: flex-end;
  }
  .retro-quick-btns {
    padding: 8px 16px;
    background: #f1f3f4;
    border-top: 1px solid #e8eaed;
    display: flex;
    gap: 8px;
    flex-wrap: wrap;
    font-size: 12px;
  }
  .quick-chip {
    background: #fff;
    border: 1px solid #dadce0;
    border-radius: 16px;
    padding: 4px 10px;
    cursor: pointer;
    color: #3c4043;
  }
  .quick-chip:hover { background: #e8f0fe; border-color: #1a73e8; color: #1a73e8; }
</style>

<div class="retro-chat-box">
  <div class="retro-chat-header">
    <div>
      <strong style="font-size:15px; color:#1a73e8;">🔬 Gemini Retrosynthesis Literature & Conditions Assistant</strong>
      <div style="font-size:12px; color:#5f6368;">Context loaded with your Target SMILES & Top 10 AiZynthFinder Routes</div>
    </div>
    <div style="display:flex; align-items:center; gap:10px;">
      <label style="font-size:12px; font-weight:600; color:#3c4043;">Model:</label>
      <select id="gemini-model-select" style="padding:5px 8px; border-radius:5px; border:1px solid #ccc; font-size:13px;">
        <option value="flash-3.8" selected>flash-3.8 (Default)</option>
        <option value="gemini-3-flash-preview">gemini-3-flash-preview</option>
        <option value="gemini-3.1-pro-preview">gemini-3.1-pro-preview</option>
        <option value="gemini-2.5-flash">gemini-2.5-flash</option>
        <option value="gemini-2.5-pro">gemini-2.5-pro</option>
      </select>
      <label style="font-size:12px; display:flex; align-items:center; gap:4px; cursor:pointer; color:#3c4043;">
        <input type="checkbox" id="gemini-search-toggle" checked /> 📚 Literature Search
      </label>
      <button id="gemini-clear-btn" style="padding:5px 10px; border:1px solid #ccc; background:#fff; border-radius:5px; cursor:pointer; font-size:12px;">
        🗑️ Clear Chat
      </button>
    </div>
  </div>

  <div id="retro-chat-messages" class="retro-chat-messages">
    <div class="msg-model">
      👋 <b>Ready!</b> I have all <b>Top 10 AiZynthFinder routes</b> for your target molecule loaded in memory.
      Ask me to search the literature for exact reaction conditions, catalysts, yields, or patent procedures for any route or step below.
    </div>
  </div>

  <div class="retro-quick-btns">
    <span style="color:#5f6368; font-weight:600; align-self:center;">Quick prompts:</span>
    <button class="quick-chip" onclick="fillPrompt('Perform a literature search for the exact reaction conditions, reagents, yields, and references for each step of Route #1.')">Route #1 Literature Conditions</button>
    <button class="quick-chip" onclick="fillPrompt('Perform a literature search for the exact reaction conditions and references for Route #6 (Suzuki coupling, SNAr, and ester hydrolysis).')">Route #6 Conditions & Refs</button>
    <button class="quick-chip" onclick="fillPrompt('Compare Route #1 (Suzuki coupling) vs Route #2 (Hantzsch thiazole cyclization) in the published literature and patents for this compound. Which has higher yield and better scalability?')">Compare Route #1 vs Route #2</button>
  </div>

  <div class="retro-chat-input-area">
    <textarea id="retro-chat-input" rows="2" placeholder="Ask about reaction conditions, catalysts, literature precedents, or side-reactions (Press Enter to send, Shift+Enter for newline)..."
              style="flex:1; padding:10px; border:1px solid #ccc; border-radius:6px; font-family:inherit; font-size:14px; resize:vertical;"></textarea>
    <button id="retro-send-btn" style="background:#1a73e8; color:white; border:none; padding:10px 18px; border-radius:6px; font-weight:bold; cursor:pointer; height:42px;">
      Send ➤
    </button>
  </div>
</div>

<script>
function fillPrompt(text) {
  const input = document.getElementById("retro-chat-input");
  input.value = text;
  input.focus();
}

async function sendRetroMessage() {
  const inputEl = document.getElementById("retro-chat-input");
  const msgContainer = document.getElementById("retro-chat-messages");
  const modelSelect = document.getElementById("gemini-model-select").value;
  const useSearch = document.getElementById("gemini-search-toggle").checked;
  const sendBtn = document.getElementById("retro-send-btn");

  const text = inputEl.value.trim();
  if (!text) return;

  // Append user bubble
  const userDiv = document.createElement("div");
  userDiv.className = "msg-user";
  userDiv.innerText = text;
  msgContainer.appendChild(userDiv);
  inputEl.value = "";

  // Append loading bubble
  const loadingDiv = document.createElement("div");
  loadingDiv.className = "msg-model";
  loadingDiv.innerHTML = "<i>🔎 Searching chemical literature & generating response with <b>" + modelSelect + "</b>...</i>";
  msgContainer.appendChild(loadingDiv);
  msgContainer.scrollTop = msgContainer.scrollHeight;
  sendBtn.disabled = true;

  try {
    const result = await google.colab.kernel.invokeFunction(
      "notebook.gemini_retro_chat",
      [text, modelSelect, useSearch],
      {}
    );
    const data = result.data["application/json"];
    if (data.error) {
      loadingDiv.innerHTML = "<span style='color:#d93025;'>⚠️ " + data.error + "</span>";
    } else {
      loadingDiv.innerHTML = data.html;
    }
  } catch (err) {
    loadingDiv.innerHTML = "<span style='color:#d93025;'>⚠️ Error communicating with Colab kernel: " + err + "</span>";
  } finally {
    sendBtn.disabled = false;
    msgContainer.scrollTop = msgContainer.scrollHeight;
  }
}

document.getElementById("retro-send-btn").onclick = sendRetroMessage;
document.getElementById("retro-chat-input").addEventListener("keydown", function(e) {
  if (e.key === "Enter" && !e.shiftKey) {
    e.preventDefault();
    sendRetroMessage();
  }
});

document.getElementById("gemini-clear-btn").onclick = async function() {
  await google.colab.kernel.invokeFunction("notebook.gemini_retro_reset", [], {});
  document.getElementById("retro-chat-messages").innerHTML =
    '<div class="msg-model">🧹 <b>Chat history cleared.</b> Ask a new question about any of your retrosynthetic routes!</div>';
};
</script>
"""

IPython.display.display(IPython.display.HTML(chat_ui_html))


# Bibliography

_Genheden S, Thakkar A, Chadimova V, et al (2020) AiZynthFinder: a fast, robust and flexible open-source software for retrosynthetic planning. J. Cheminf. https://doi.org/10.1186/s13321-020-00472-1 ([GitHub](https://github.com/MolecularAI/aizynthfinder) & [Documentation](https://molecularai.github.io/aizynthfinder/html/index.html))_